# Assignment -- Cedar Grove Public Library: Checkouts

**5 problems, basic -> medium.** Same core skills as class (pandas: loading, cleaning,
grouping; `requests`: calling a real public API and parsing JSON) applied to a brand-new
scenario and dataset.

**Estimated time:** 45-60 minutes

## The scenario

Cedar Grove Public Library tracks every book checkout in `data/checkouts.csv`: who
checked out which book, when it was due back, when (if ever) it was returned, and any
late fee charged. The head librarian has five questions.

## Setup

```bash
pip install -r requirements.txt
python scripts/generate_checkouts_data.py   # only if data/checkouts.csv isn't already there
jupyter lab Assignment_Starter.ipynb
```

Problem 4 calls a real public API ([Open Library](https://openlibrary.org/)) and needs an
open internet connection; its function includes a fallback so the assignment stays
completable even if that call fails.

Each problem names the exact variable you need to produce, and most have a small
"check yourself" cell with `assert` statements right after.

---
## Problem 1 (basic) -- Load and get oriented

Load `data/checkouts.csv` into `checkouts_df`, parsing `checkout_date`, `due_date`, and
`return_date` as real dates. Then answer: how many checkouts are there in total, and how
many have never been returned (i.e. `return_date` is missing)? Store these two numbers as
`n_total_checkouts` and `n_still_checked_out`.

In [2]:
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 20)

In [3]:
# TODO: load data/checkouts.csv, parsing the three date columns
checkouts_df = pd.read_csv("data/checkouts.csv", parse_dates = ["checkout_date","due_date","return_date"])
checkouts_df.dtypes
# TODO: how many rows total, and how many are missing return_date?
n_total_checkouts = len(checkouts_df)
n_still_checked_out = checkouts_df["return_date"].isnull().sum()

print(f"{n_total_checkouts} total checkouts, {n_still_checked_out} still checked out")

160 total checkouts, 48 still checked out


In [26]:
# Check yourself
assert n_total_checkouts == len(checkouts_df)
assert n_still_checked_out == checkouts_df["return_date"].isna().sum()
assert n_still_checked_out < n_total_checkouts
print("Looks good.")

Looks good.


---
## Problem 2 (basic-medium) -- Clean the data, the right way for each column

A missing `return_date` here does **not** mean bad data -- it means the book is still
checked out, which is a completely normal, valid state. So this time, don't drop those
rows or invent a fake return date for them; instead, capture "returned or not" explicitly.

Build `checkouts_clean` from `checkouts_df` with:

- a new boolean column `is_returned`, `True` where `return_date` is present and `False`
  where it's missing
- `late_fee` missing filled with `0` (missing here means no fee was ever charged -- either
  the book came back on time, or a librarian waived the fee)

In [31]:
checkouts_clean = checkouts_df.copy()

# TODO: add an is_returned boolean column (True where return_date is present)
checkouts_clean["is_returned"] = checkouts_clean["return_date"].notna()

# TODO: fill missing late_fee with 0
checkouts_clean["late_fee"] = checkouts_clean["late_fee"].fillna(0)

checkouts_clean

,checkout_id,member_id,book_title,genre,checkout_date,due_date,return_date,late_fee,is_returned
0,CHK-3001,MEM-0014,The Great Gatsby,Classic Fiction,2026-06-12,2026-07-03,2026-07-03,0.0,True
1,CHK-3002,MEM-0034,Jane Eyre,Gothic,2026-07-19,2026-08-09,NaT,0.0,False
2,CHK-3003,MEM-0060,The Catcher in the Rye,Classic Fiction,2026-01-02,2026-01-23,NaT,0.0,False
3,CHK-3004,MEM-0051,The Hobbit,Adventure,2026-06-16,2026-07-07,NaT,0.0,False
4,CHK-3005,MEM-0028,The Catcher in the Rye,Classic Fiction,2026-04-01,2026-04-22,2026-04-22,0.0,True
...,...,...,...,...,...,...,...,...,...
155,CHK-3156,MEM-0038,Brave New World,Dystopian,2026-02-13,2026-03-06,NaT,0.0,False
156,CHK-3157,MEM-0047,1984,Dystopian,2026-06-23,2026-07-14,2026-07-14,0.0,True
157,CHK-3158,MEM-0027,Moby Dick,Adventure,2026-02-14,2026-03-07,2026-03-16,0.0,True
158,CHK-3159,MEM-0055,1984,Dystopian,2026-06-29,2026-07-20,2026-07-20,0.0,True


In [32]:
checkouts_clean.dtypes

checkout_id              object
member_id                object
book_title               object
genre                    object
checkout_date    datetime64[ns]
due_date         datetime64[ns]
return_date      datetime64[ns]
late_fee                float64
is_returned                bool
dtype: object

In [33]:
# Check yourself
assert checkouts_clean["late_fee"].isna().sum() == 0
assert checkouts_clean["is_returned"].dtype == bool
assert checkouts_clean["is_returned"].sum() == checkouts_clean["return_date"].notna().sum()
print("Looks good:", checkouts_clean["is_returned"].value_counts().to_dict())

Looks good: {True: 112, False: 48}


---
## Problem 3 (medium) -- Which genre racks up the most late fees?

Using only **returned** books (`is_returned == True`), compute the average `late_fee` per
`genre`, sorted from highest to lowest, as `avg_late_fee_by_genre`.

In [6]:
# TODO: filter to returned books only, then average late_fee by genre, sorted descending

returned_only = checkouts_clean[checkouts_clean["is_returned"] == True]
avg_late_fee_by_genre = checkouts_clean.groupby("genre")["late_fee"].mean().sort_values(ascending=False)


avg_late_fee_by_genre

genre
Dystopian             0.491667
Historical Fiction    0.446429
Adventure             0.383929
Gothic                0.250000
Classic Fiction       0.218750
Name: late_fee, dtype: float64

In [34]:
# Check yourself
assert len(avg_late_fee_by_genre) == checkouts_clean["genre"].nunique()
assert avg_late_fee_by_genre.is_monotonic_decreasing
print("Looks good -- worst genre for late fees:", avg_late_fee_by_genre.idxmax())

Looks good -- worst genre for late fees: Dystopian


---
## Problem 4 (basic-medium) -- Look up each book with a real public API

The library's own data doesn't say who wrote each book or when it was first published.
[Open Library](https://openlibrary.org/) has a free, no-API-key search endpoint that does:

`https://openlibrary.org/search.json?q={title}`

The response's `"docs"` list holds the matches; the first one is normally the best match.
Each doc has `author_name` (a **list** of strings) and `first_publish_year`.

Write `get_book_facts(title)`, returning `{"author": ..., "first_publish_year": ...}` for
the top match. Wrap the request in a `try`/`except` and fall back to
`BACKUP_BOOK_FACTS[title]` (given below) if the call fails or the response looks wrong --
same resilience pattern as the real-API-with-a-fallback from class.

In [7]:
# Known-correct backup facts (classroom fallback only, used if the live API call fails).
BACKUP_BOOK_FACTS = {
    "Pride and Prejudice": {"author": "Jane Austen", "first_publish_year": 1813},
    "To Kill a Mockingbird": {"author": "Harper Lee", "first_publish_year": 1960},
    "The Great Gatsby": {"author": "F. Scott Fitzgerald", "first_publish_year": 1925},
    "The Catcher in the Rye": {"author": "J. D. Salinger", "first_publish_year": 1951},
    "1984": {"author": "George Orwell", "first_publish_year": 1949},
    "Brave New World": {"author": "Aldous Huxley", "first_publish_year": 1932},
    "Frankenstein": {"author": "Mary Shelley", "first_publish_year": 1818},
    "Jane Eyre": {"author": "Charlotte Bronte", "first_publish_year": 1847},
    "Moby Dick": {"author": "Herman Melville", "first_publish_year": 1851},
    "The Hobbit": {"author": "J. R. R. Tolkien", "first_publish_year": 1937},
    "War and Peace": {"author": "Leo Tolstoy", "first_publish_year": 1869},
    "Crime and Punishment": {"author": "Fyodor Dostoevsky", "first_publish_year": 1866},
}

OPEN_LIBRARY_API = "https://openlibrary.org/search.json"

In [10]:
import requests

def get_book_facts(title):
    # TODO: GET OPEN_LIBRARY_API with params={"q": title}, raise_for_status(), take
    # response.json()["docs"][0], and return {"author": ..., "first_publish_year": ...}
    # (author_name is a list -- use its first item). On any RequestException, KeyError,
    # or IndexError, fall back to BACKUP_BOOK_FACTS[title] instead of crashing.
    try:
        response = requests.get(f"{OPEN_LIBRARY_API}",
                                params={"q": title} )
        response.raise_for_status()  # raises an exception here if the call failed -- fail loudly, not silently

        data = response.json()
        doc = data["docs"][0]

        return {
            "author": doc["author_name"][0],
            "first_publish_year": doc["first_publish_year"]
        }

    except (requests.exceptions.RequestException, KeyError, IndexError):
        return BACKUP_BOOK_FACTS[title]
    
get_book_facts("1984")


{'author': 'George Orwell', 'first_publish_year': 1949}

Now call it for every distinct title in the library's catalog and assemble `book_facts_df`,
indexed by `book_title`, with columns `author` and `first_publish_year`.

In [22]:
# TODO: call get_book_facts for every unique book_title in checkouts_clean and
# build book_facts_df indexed by book_title
records = {}
# your loop here
for title in checkouts_clean["book_title"].unique():
    records[title] = get_book_facts(title)

book_facts_df = pd.DataFrame(records).T
book_facts_df.index.name = "book_title"
book_facts_df

,author,first_publish_year
book_title,,
The Great Gatsby,F. Scott Fitzgerald,1920
Jane Eyre,Charlotte Brontë,1847
The Catcher in the Rye,J. D. Salinger,1945
The Hobbit,J.R.R. Tolkien,1937
Crime and Punishment,Фёдор Достоевский,1866
War and Peace,Лев Толстой,1864
To Kill a Mockingbird,Harper Lee,1960
Moby Dick,Herman Melville,1851
Brave New World,Aldous Huxley,1932


In [23]:
# Check yourself
assert len(book_facts_df) == checkouts_clean["book_title"].nunique()
assert set(["author", "first_publish_year"]).issubset(book_facts_df.columns)
print("Looks good:", book_facts_df.shape)

Looks good: (12, 2)


---
## Problem 5 (medium) -- Which author costs the library the most in late fees?

Merge `book_facts_df` into `checkouts_clean` (on `book_title`), then compute total
`late_fee` collected **per author**, sorted descending, as `late_fee_by_author`.

In [24]:
# TODO: merge checkouts_clean with book_facts_df on book_title (book_facts_df's
# index is the title, so you'll need it as a column first -- see reset_index()),
# then total late_fee by author, sorted descending
checkouts_with_author = checkouts_clean.merge(book_facts_df.reset_index(),
                                              on="book_title",
                                              how="left")
late_fee_by_author = (checkouts_with_author.groupby("author")["late_fee"].sum().sort_values(ascending = False))
late_fee_by_author

author
Aldous Huxley          7.75
George Orwell          7.00
Herman Melville        6.50
Фёдор Достоевский      6.50
Лев Толстой            6.00
J. D. Salinger         5.75
Charlotte Brontë      4.25
J.R.R. Tolkien         4.25
F. Scott Fitzgerald    2.25
Mary Shelley           2.25
Harper Lee             1.25
Jane Austen            1.25
Name: late_fee, dtype: float64

In [25]:
# Check yourself
assert len(checkouts_with_author) == len(checkouts_clean)
assert late_fee_by_author.is_monotonic_decreasing
print("Looks good -- costliest author:", late_fee_by_author.idxmax())

Looks good -- costliest author: Aldous Huxley
